# Testing: Synthetic Data

The `synthetic` module allows us to create artificial datasets with known velocities, extents etc. which we can then compare with those estimated by THUNER.

## Geographic Coordinates

In [1]:
"""Synthetic data demo/test."""

%load_ext autoreload
%autoreload 2
import xarray as xr
from pathlib import Path
import shutil
import numpy as np
import thuner.data as data
import thuner.default as default
import thuner.track.track as track
import thuner.option as option
import thuner.analyze as analyze
import thuner.data.synthetic as synthetic


Welcome to the Thunderstorm Event Reconnaissance (THUNER) package 
v0.0.16! This package is still in testing and development. Please 
visit github.com/THUNER-project/THUNER for examples, and to report 
issues or contribute.
 
THUNER is a flexible toolkit for performing multi-feature detection, 
tracking, tagging and analysis of events within meteorological datasets. 
The intended application is to convective weather events. For examples 
and instructions, see https://github.com/THUNER-project/THUNER and 
https://thuner.readthedocs.io/en/latest/. If you use THUNER in your 
research, consider citing the following papers;

Short et al. (2023), doi: 10.1175/MWR-D-22-0146.1
Raut et al. (2021), doi: 10.1175/JAMC-D-20-0119.1
Fridlind et al. (2019), doi: 10.5194/amt-12-2979-2019
Whitehall et al. (2015), doi: 10.1007/s12145-014-0181-3
Dixon and Wiener (1993), doi: 10.1175/1520-0426(1993)010<0785:TTITAA>2.0.CO;2
Leese et al. (1971), doi: 10.1175/1520-0450(1971)010<0118:AATFOC>2.0.CO;2



In [ ]:
# Set a flag for whether or not to remove existing output directories
remove_existing_outputs = True

# Parent directory for saving outputs
base_local = Path.home() / "THUNER_output"
start = "2005-11-13T00:00:00"
end = "2005-11-13T01:00:00"

output_parent = base_local / "runs/synthetic/geographic"
if output_parent.exists() and remove_existing_outputs:
    shutil.rmtree(output_parent)
options_directory = output_parent / "options"
options_directory.mkdir(parents=True, exist_ok=True)

# Create a grid
lat = np.arange(-14, -6 + 0.025, 0.025).tolist()
lon = np.arange(128, 136 + 0.025, 0.025).tolist()
grid_options = option.grid.GridOptions(name="geographic", latitude=lat, longitude=lon)
grid_options.to_json(options_directory / "grid.json")

# Initialize synthetic objects
starting_objects = []
for i in range(5):
    major = 2 * (7 + 4 * i)  # full axis length in km
    obj = synthetic.EllipsoidObject(
        time=start,
        center_latitude=np.mean(lat),
        center_longitude=lon[(i + 1) * len(lon) // 6],
        direction=-np.pi / 4 + i * np.pi / 8,
        speed=30 - 4 * i,
        major=major,
        minor=0.4 * major,
        orientation=0.25 * np.pi + i * np.pi / 8,
    )
    starting_objects.append(obj)
# Create data options dictionary
synthetic_options = data.synthetic.SyntheticOptions(objects=starting_objects)
data_options = option.data.DataOptions(datasets=[synthetic_options])
data_options.to_json(options_directory / "data.json")

track_options = default.track.synthetic_track()
track_options.to_json(options_directory / "track.json")

# Create the display_options dictionary
visualize_options = default.visualize.synthetic_runtime(
    options_directory / "visualize.json"
)
visualize_options.to_json(options_directory / "visualize.json")

In [3]:
visualize_options.model_dump()

{'type': 'RuntimeOptions',
 'objects': {'convective': {'type': 'ObjectRuntimeOptions',
   'parent_local': PosixPath('/home/ewan/THUNER_output/runs/synthetic/geographic/options/visualize.json'),
   'style': 'presentation',
   'weights_filepath': None,
   'name': 'convective',
   'figures': [{'type': 'FigureOptions',
     'name': 'match',
     'function': 'thuner.visualize.runtime.visualize_tint_match',
     'style': 'presentation',
     'animate': True,
     'single_color': False,
     'template': None}],
   'animate': True,
   'single_color': False}}}

In [4]:
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    np.timedelta64(10, "m"),
)
track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=visualize_options,
    output_directory=output_parent,
)

2026-06-04 17:42:45,308 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/geographic.
2026-06-04 17:42:45,311 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-04 17:42:45,313 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.


2026-06-04 17:42:47,625 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 17:42:47,627 - thuner.track.track - INFO - Tracking convective.
2026-06-04 17:42:47,635 - thuner.utils - INFO - Compiling thuner.detect.steiner.steiner_scheme with Numba. Please wait.
2026-06-04 17:42:47,854 - thuner.match.match - INFO - Matching convective objects.
2026-06-04 17:42:47,855 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-04 17:42:47,860 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-04 17:42:59,391 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-04 17:42:59,392 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-04 17:43:01,283 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-04 17:43:01,284 - thuner.track.track - INFO - Tracking convective.
2026-06-04 17:43:01,288 - thuner.write.mask - INFO - Writing convective ma

![THUNER applied to synthetic data.](https://raw.githubusercontent.com/THUNER-project/THUNER/refs/heads/main/gallery/synthetic.gif)

In [7]:
ground_truth = analyze.synthetic.write_ground_truth(
    output_parent, data_options=data_options, times=times
)
ground_truth["synthetic"]

2026-06-04 17:51:34,938 - thuner.analyze.synthetic - INFO - Wrote ground truth for synthetic.


latitude  longitude     u     v  horizontal_radius  \
time                id                                                       
2005-11-13 00:00:00 0   -10.0000   129.3250 -21.2  21.2                7.0   
                    1   -10.0000   130.6750  -9.9  24.0               11.0   
                    2   -10.0000   132.0250   0.0  22.0               15.0   
                    3   -10.0000   133.3500   6.9  16.6               19.0   
                    4   -10.0000   134.7000   9.9   9.9               23.0   
2005-11-13 00:10:00 0    -9.8849   129.2090 -21.2  21.2                7.0   
                    1    -9.8697   130.6206  -9.9  24.0               11.0   
                    2    -9.8807   132.0250   0.0  22.0               15.0   
                    3    -9.9098   133.3877   6.9  16.6               19.0   
                    4    -9.9463   134.7542   9.9   9.9               23.0   
2005-11-13 00:20:00 0    -9.7698   129.0930 -21.2  21.2                7.0   
                    1    -9.7394   130.5662  -9.9  24.0               11.0   
                    2    -9.7613   132.0250   0.0  22.0               15.0   
                    3    -9.8196   133.4254   6.9  16.6               19.0   
                    4    -9.8926   134.8083   9.9   9.9               23.0   
2005-11-13 00:30:00 0    -9.6546   128.9771 -21.2  21.2                7.0   
                    1    -9.6090   130.5118  -9.9  24.0               11.0   
                    2    -9.6420   132.0250   0.0  22.0               15.0   
                    3    -9.7293   133.4630   6.9  16.6               19.0   
                    4    -9.8389   134.8624   9.9   9.9               23.0   
2005-11-13 00:40:00 0    -9.5394   128.8613 -21.2  21.2                7.0   
                    1    -9.4787   130.4575  -9.9  24.0               11.0   
                    2    -9.5226   132.0250   0.0  22.0               15.0   
                    3    -9.6391   133.5006   6.9  16.6               19.0   
                    4    -9.7851   134.9166   9.9   9.9               23.0   
2005-11-13 00:50:00 0    -9.4241   128.7456 -21.2  21.2                7.0   
                    1    -9.3484   130.4033  -9.9  24.0               11.0   
                    2    -9.4033   132.0250   0.0  22.0               15.0   
                    3    -9.5489   133.5382   6.9  16.6               19.0   
                    4    -9.7314   134.9707   9.9   9.9               23.0   
2005-11-13 01:00:00 0    -9.3088   128.6299 -21.2  21.2                7.0   
                    1    -9.2180   130.3491  -9.9  24.0               11.0   
                    2    -9.2839   132.0250   0.0  22.0               15.0   
                    3    -9.4587   133.5758   6.9  16.6               19.0   
                    4    -9.6776   135.0247   9.9   9.9               23.0   

                        eccentricity  orientation  intensity  
time                id                                        
2005-11-13 00:00:00 0            0.4     0.785398         50  
                    1            0.4     1.178097         50  
                    2            0.4     1.570796         50  
                    3            0.4     1.963495         50  
                    4            0.4     2.356194         50  
2005-11-13 00:10:00 0            0.4     0.785398         50  
                    1            0.4     1.178097         50  
                    2            0.4     1.570796         50  
                    3            0.4     1.963495         50  
                    4            0.4     2.356194         50  
2005-11-13 00:20:00 0            0.4     0.785398         50  
                    1            0.4     1.178097         50  
                    2            0.4     1.570796         50  
                    3            0.4     1.963495         50  
                    4            0.4     2.356194         50  
2005-11-13 00:30:00 0            0.4     0.785398         50  
   

In [6]:
central_latitude = -10
central_longitude = 132

y = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()
x = np.arange(-400e3, 400e3 + 2.5e3, 2.5e3).tolist()

grid_options = option.grid.GridOptions(
    name="cartesian",
    x=x,
    y=y,
    central_latitude=central_latitude,
    central_longitude=central_longitude,
)
grid_options.to_json(options_directory / "grid.json")

2026-06-03 23:37:54,422 - thuner.option.grid - WARNING - altitude not specified. Using default altitudes.


In [ ]:
output_parent = base_local / "runs/synthetic/cartesian"
if output_parent.exists() & remove_existing_outputs:
    shutil.rmtree(output_parent)
    
times = np.arange(
    np.datetime64(start),
    np.datetime64(end) + np.timedelta64(10, "m"),
    +np.timedelta64(10, "m"),
)

track.track(
    times=times,
    data_options=data_options,
    grid_options=grid_options,
    track_options=track_options,
    visualize_options=None,
    output_directory=output_parent,
)

2026-06-03 23:37:55,143 - thuner.track.track - INFO - Beginning thuner tracking. Saving output to /home/ewan/THUNER_output/runs/synthetic/cartesian.
2026-06-03 23:37:55,145 - thuner.track.track - INFO - Processing 2005-11-13T00:00:00.
2026-06-03 23:37:55,146 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:00:00.


2026-06-03 23:37:56,331 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:56,331 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - Matching convective objects.
2026-06-03 23:37:56,360 - thuner.match.match - INFO - No current mask, or no objects in current mask.
2026-06-03 23:37:56,363 - thuner.visualize.runtime - INFO - Creating runtime visualization figures.
2026-06-03 23:37:58,238 - thuner.track.track - INFO - Processing 2005-11-13T00:10:00.
2026-06-03 23:37:58,239 - thuner.data.synthetic.options - INFO - Updating synthetic dataset for 2005-11-13T00:10:00.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Processing hierarchy level 0.
2026-06-03 23:37:59,290 - thuner.track.track - INFO - Tracking convective.
2026-06-03 23:37:59,293 - thuner.write.mask - INFO - Writing convective masks to /home/ewan/THUNER_output/runs/synthetic/cartesian/output.zarr::masks/convective.
2026-06-03 23:37:59,327 - thuner